<a href="https://colab.research.google.com/github/louistrue/DB-1/blob/main/DB1_W10_Demo_IFC_Mengen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DB1 – Woche 10: Demo – Mengenermittlung aus IFC
**Live-Demo: Gleiche Logik, echtes BIM-Modell**

---

In eurem Notebook habt ihr Mengen aus einer **JSON-Datei** ermittelt. Hier zeige ich euch dasselbe Muster auf einer **echten IFC-Datei**.

| Im Studierenden-Notebook        | In dieser Demo                  |
|---------------------------------|----------------------------------|
| `bruecke = json.load(...)`      | `ifc = ifcopenshell.open(...)`   |
| `for stab in bruecke["staebe"]` | `for el in ifc.by_type("IfcBeam")` |
| `stab["material"]`              | `material via util.element`      |
| `stab["laenge_m"]`              | `Qto_BeamBaseQuantities.Length`  |

> **Das `for`-Skelett bleibt gleich, nur die Datenquelle ändert sich.**

IFC ist der offene Standard für BIM. Mehr dazu in **DB2**. Heute reicht: IFC = strukturierte Daten, gleiche Iterations-Logik wie JSON, nur reicher.

---

## Setup

1. Falls noch nicht installiert: `!pip install ifcopenshell plotly`
2. Eine IFC-Datei aus Moodle herunterladen und in Colab hochladen (Sidebar → Files → Upload).
3. Den Pfad unten anpassen.

In [ ]:
# Setup - bei Bedarf installieren
!pip install -q ifcopenshell plotly


In [ ]:
import ifcopenshell
import ifcopenshell.util.element
import plotly.graph_objects as go
from collections import defaultdict

import requests

# URL of the raw IFC file on GitHub
url = "https://github.com/louistrue/learn-ifc/raw/main/Modelle/BFH-25/03_BIMcollab_Example_STR.ifc"

# Extract filename from URL
filename = url.split('/')[-1]

# Download the file
response = requests.get(url)
response.raise_for_status() # Raise an exception for bad status codes

with open(filename, 'wb') as f:
    f.write(response.content)

print(f"Downloaded {filename}")

ifc = ifcopenshell.open(filename)

# Datei-Übersicht
project = ifc.by_type("IfcProject")[0] if ifc.by_type("IfcProject") else None
print(f"IFC-Schema:         {ifc.schema}")
if project:
    print(f"Projekt:            {project.Name}")
print(f"Anzahl Bauteile:    {len(ifc.by_type('IfcProduct'))}")
print(f"Anzahl Materialien: {len(ifc.by_type('IfcMaterial'))}")


---
## Teil 1 – Anzahl Bauteile pro IfcClass

Im Studierenden-Notebook habt ihr `for stab in bruecke["staebe"]` über *unsere* Klasse iteriert. Eine IFC enthält viele Klassen (`IfcBeam`, `IfcColumn`, `IfcWall`, `IfcSlab`, ...).

Wir zählen pro Klasse – dasselbe Muster wie *Anzahl pro Material*.

In [ ]:
# Welche Klassen kommen vor? (Strukturteile)
strukturklassen = ["IfcBeam", "IfcColumn", "IfcMember", "IfcSlab", "IfcWall",
                   "IfcRailing", "IfcStair", "IfcRoof", "IfcCovering"]

anzahl_pro_klasse = {}
for klasse in strukturklassen:
    elemente = ifc.by_type(klasse)
    if elemente:
        anzahl_pro_klasse[klasse] = len(elemente)

print("Bauteile pro IFC-Klasse:")
for klasse, n in sorted(anzahl_pro_klasse.items(), key=lambda x: -x[1]):
    print(f"  {klasse:<15} {n}")


---
## Teil 2 – Material pro Bauteil

In IFC ist Material nicht direkt ein Attribut, sondern über eine **Relation** verknüpft (`IfcRelAssociatesMaterial`). `ifcopenshell.util.element.get_material()` versteckt diese Komplexität.

In [ ]:
# Material pro Bauteil ermitteln (über Helper-Funktion)

def material_name(element):
    """Gibt einen Material-String zurück, oder 'unbekannt'."""
    mat = ifcopenshell.util.element.get_material(element)
    if mat is None:
        return "unbekannt"
    # Material kann verschiedene Formen haben: einfach, Set, Liste...
    if hasattr(mat, "Name") and mat.Name:
        return mat.Name
    if hasattr(mat, "ForLayerSet"):  # MaterialLayerSetUsage
        layers = mat.ForLayerSet.MaterialLayers
        return ", ".join(L.Material.Name for L in layers if L.Material)
    if hasattr(mat, "Materials"):    # MaterialList
        return ", ".join(m.Name for m in mat.Materials if m.Name)
    return str(mat)

# Beispiel: erste 5 Bauteile mit Material
print("Beispiel-Bauteile mit Material:")
for el in ifc.by_type("IfcProduct")[:8]:
    if el.is_a("IfcProject") or el.is_a("IfcSite") or el.is_a("IfcBuilding") or el.is_a("IfcBuildingStorey"):
        continue
    print(f"  {el.is_a():<12} {el.Name or '(unbenannt)':<25} → {material_name(el)}")


---
## Teil 3 – Mengen aus Base Quantities

In gut gepflegten IFC-Modellen liegen Mengen in standardisierten Property-Sets, z.B. `Qto_BeamBaseQuantities` mit `Length`, `NetVolume`, `GrossVolume`.

`ifcopenshell.util.element.get_psets()` liest alle Psets/Qtos auf einmal aus.

In [ ]:
# Mengen pro Bauteil ermitteln

def get_quantity(element, *names):
    """Sucht in allen Qto_*BaseQuantities nach den genannten Property-Namen.
    Gibt den ersten Treffer zurück, sonst None."""
    psets = ifcopenshell.util.element.get_psets(element, qtos_only=True)
    for qto_name, props in psets.items():
        for n in names:
            if n in props and props[n] is not None:
                return props[n]
    return None

# Beispiel-Output für die ersten Strukturteile
print(f"{'Klasse':<12} {'Name':<25} {'Material':<22} {'L [m]':>8} {'V [m³]':>10}")
print("-" * 80)
strukturteile = []
for klasse in anzahl_pro_klasse:
    strukturteile.extend(ifc.by_type(klasse))

for el in strukturteile[:10]:
    L = get_quantity(el, "Length", "NominalLength")
    V = get_quantity(el, "NetVolume", "GrossVolume", "Volume")
    L_str = f"{L:>8.2f}" if L is not None else "    n.v."
    V_str = f"{V:>10.4f}" if V is not None else "      n.v."
    name = (el.Name or "")[:24]
    print(f"{el.is_a():<12} {name:<25} {material_name(el)[:21]:<22} {L_str} {V_str}")


---
## Teil 4 – Aggregation: Mengen pro Material

Jetzt das gleiche Aggregations-Muster wie im Studierenden-Notebook: **alle Strukturteile durchgehen, Material gruppieren, Längen und Volumen aufsummieren.**

```python
# Studierenden-Notebook (W10 Aufgabe 5)
for stab in gueltige_staebe:
    m = stab["material"]
    laenge_pro_material[m] += stab["laenge_m"]
```
↓ exakt dasselbe auf IFC ↓

In [ ]:
# Aggregation pro Material - das vertraute for-Skelett

laenge_pro_material  = defaultdict(float)
volumen_pro_material = defaultdict(float)
anzahl_pro_material  = defaultdict(int)

for el in strukturteile:
    m = material_name(el)
    L = get_quantity(el, "Length", "NominalLength") or 0
    V = get_quantity(el, "NetVolume", "GrossVolume", "Volume") or 0

    anzahl_pro_material[m]  += 1
    laenge_pro_material[m]  += L
    volumen_pro_material[m] += V

# Tabellen-Ausgabe
print(f"{'Material':<25} {'Anzahl':>7} {'L [m]':>10} {'V [m³]':>10}")
print("-" * 56)
for m in sorted(anzahl_pro_material):
    print(f"{m[:24]:<25} {anzahl_pro_material[m]:>7} "
          f"{laenge_pro_material[m]:>10.2f} {volumen_pro_material[m]:>10.4f}")


---
## Teil 5 – Plotly: Volumen pro Material

Wieder dieselbe Visualisierung wie im Studierenden-Notebook.

In [ ]:
# Bar-Chart Volumen pro Material

materialien = list(volumen_pro_material.keys())
volumina    = list(volumen_pro_material.values())

fig = go.Figure(data=[
    go.Bar(x=materialien, y=volumina,
           text=[f"{v:.3f} m³" for v in volumina],
           textposition="outside")
])
fig.update_layout(
    title=f"Volumen pro Material ({filename})",
    xaxis_title="Material",
    yaxis_title="Volumen [m³]",
    height=420,
)
fig.show()

---
## Was war hier neu, was war gleich?

**Gleich**:
- Die `for`-Schleife
- `dict.get()` / `defaultdict` für Aggregation
- Plotly für Visualisierung
- Die Idee: Iterieren → Attribut lesen → Aggregieren → Anzeigen

**Anders**:
- Daten kommen aus einer **standardisierten** BIM-Datei (IFC), nicht einer projektspezifischen JSON
- Material ist über eine **Relation** verknüpft, nicht direkt am Bauteil
- Mengen liegen in **Base Quantities** Property-Sets

**Take-away**: Wenn ihr in der Praxis vor einem 50 MB IFC-Modell sitzt, ist die **Frage** dieselbe wie bei eurer Brücke 2. Nur das Datenformat hat dazwischen gewachsen. **DB2** vertieft IFC und IDS – heute habt ihr das Muster gesehen.

---
*Digitales Bauen 1 | BFH AHB | Louis Trümpler*